<!--

    Gaia Data Processing and Analysis Consortium (DPAC) 
    Co-ordination Unit 9 Work Package 930
    
    (c) 2005-2025 Gaia DPAC
    
    This program is free software: you can redistribute it and/or modify
    it under the terms of the GNU General Public License as published by
    the Free Software Foundation, either version 3 of the License, or
    (at your option) any later version.

    This program is distributed in the hope that it will be useful,
    but WITHOUT ANY WARRANTY; without even the implied warranty of
    MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.  See the
    GNU General Public License for more details.

    You should have received a copy of the GNU General Public License
    along with this program.  If not, see <https://www.gnu.org/licenses/>.
    -->


This Gaia science exploitation example is built on [Dask](https://www.dask.org/). The primary user interface uses web-based Jupyter notebooks, employing Python code and a suite of third-party packages and libraries for analysis, plotting, etc. A working knowledge of Gaia data releases to-date, Python, and [Pandas](https://pandas.pydata.org/) is assumed.


The best way to familiarise yourself with the platform is to examine the example notebooks that come bundled with it. These are:

1. Start here: platform familiarisation (this notebook);
1. Data holdings: a description of the data hosted on the platform;
1. Source counts over the sky: a simple example of querying and plotting;
1. Mean proper motions over the sky: another simple example;
1. Working with Gaia XP spectra: demonstrates trawling the large XP spectral dataset, and plotting of the spectra;
1. Analyses involving cross-match surveys: using Gaia data in conjunction with optical and infrared survey data;
1. Good astrometric solutions via ML Random Forest classifier
1. Tips and tricks: some further usage tips and technical aspects of the platform including interactions and plotting.

You can find example notebooks in `demo/` in your home directory. Note that you will need to copy these (`right-click` > `copy`) to edit or run them. 

Newcomers are advised to work through at least the first four notebooks before attempting to use the system.

This and all the other example notebooks itemised above are Jupyter Notebooks. Text, Python code, and plots can be freely mixed in the notebook in paragraphs, or cells. For example, to see the [markdown code](https://jupyter-notebook.readthedocs.io/en/stable/examples/Notebook/Working%20With%20Markdown%20Cells.html) for this cell simply double click anywhere; execute the cell with `Ctrl` + (`Alt` + )`Shift` or the "play" button in the panel above to hide the code leaving only the output.  

To save a notebook locally, select `File` > `Download` in the menu at the top, or right-click the notebook in the file viewer on the left and select "Download". This will create a local file with extension ".ipynb". To import a previously saved note (or indeed one provisioned externally) click on the upload icon in the top left of the UI and select a file from your local file system.

Please note that:
* You are strongly advised to save locally all your notebook edits at frequent intervals (see above). Preservation of the notebook content of the platform can be managed on a best-efforts basis only. While we will endeavour to restore fully in the event of loss of the deployed system or Cloud services on which it is built, we can make no guarantee that your work will be backed up and will be recovered; 
* Important etiquette: Astroflow is a resource shared between users. Please be considerate to fellow users by logging out of the web interface when you are not using it. In particular please do not leave notebooks open and holding cached data when you are not using them - further examples of best practice are given below.  


The platform environment provides a Dask API (as an alternative to Spark) for distributed processing and scalability to high-volume data releases.  The primary distributed data container is known as a data frame, which in turn consists of multiple Pandas [DataFrames](https://pandas.pydata.org/docs/user_guide/10min.html), a richly structured object encapsulating data, descriptions and methods that is _not_ limited to simple tables with primitive column types in contrast to systems built on relational data base technology (RDBMS). 

**How to import Dask**
        
... see the next code cell. The notebook interpreter can be set for individual cells as is done here and in the next by selecting from the drop-down menu at the cell settings above (which will currently be either "Code" or "Markdown")

First you need to set up a Dask client. You will need to do this for each notebook you create.

In [ ]:
from dask.distributed import Client
c = Client('tcp://simple-scheduler.dask-operator.svc.cluster.local:8786')

In [ ]:
import dask.dataframe as dd

You can then read tables from parquet, a column-oriented storage format:

In [ ]:
import os; from pathlib import Path
ddf = dd.read_parquet(Path(os.environ['DASK_DATA_PATH_GAIA_DR3_SSD']).joinpath('GDR3_GAIA_SOURCE/'), columns=["source_id"])

To find available tables, you can list tables in `/mnt/gaia-dr3-data`. 

_Jupyter tip_: use `!` to run shell commands in notebook cells. You can also open a terminal inside the Jupyter platform via either `File` > `New` > `Terminal` or via a new tab.

In [ ]:
! ls ${DASK_DATA_PATH_GAIA_DR3_SSD}

GDR3_2MASSPSC_BEST_NEIGHBOURS	    GDR3_SSO_REFLECTANCE_SPECTRUM
GDR3_ALERTS_MIXEDIN_SOURCEIDS	    GDR3_SSO_SOURCE
GDR3_ALLWISE_BEST_NEIGHBOURS	    GDR3_TOTAL_GALACTIC_EXTINCTION_MAP
GDR3_ASTROPHYSICAL_PARAMETERS	    GDR3_TOTAL_GALACTIC_EXTINCTION_MAP_OPT
GDR3_ASTROPHYSICAL_PARAMETERS_SUPP  GDR3_VARI_AGN
GDR3_EPOCH_PHOTOMETRY		    GDR3_VARI_CEPHEID
GDR3_GAIASOURCE			    GDR3_VARI_CLASSIFIER_CLASS_DEFINITION
GDR3_GAIA_SOURCE		    GDR3_VARI_CLASSIFIER_DEFINITION
GDR3_GALAXY_CANDIDATES		    GDR3_VARI_CLASSIFIER_RESULT
GDR3_GALAXY_CATALOGUE_NAME	    GDR3_VARI_COMPACT_COMPANION
GDR3_MCMC_SAMPLES_GSP_PHOT	    GDR3_VARI_ECLIPSING_BINARY
GDR3_MCMC_SAMPLES_MSC		    GDR3_VARI_EPOCH_RADIAL_VELOCITY
GDR3_NSS_ACCELERATION_ASTRO	    GDR3_VARI_LONG_PERIOD_VARIABLE
GDR3_NSS_NON_LINEAR_SPECTRO	    GDR3_VARI_MICROLENSING
GDR3_NSS_TWO_BODY_ORBIT		    GDR3_VARI_MS_OSCILLATOR
GDR3_NSS_VIM_FL			    GDR3_VARI_PLANETARY_TRANSIT
GDR3_OA_NEURON_INFORMATION	    GDR3_VARI_RAD_VEL_STATISTICS
GDR3_OA_NEURON_XP_SPECT

The operating model of Dask as actioned via a workflow expressed in a notebook differs greatly from how Pandas works. Users should be aware of the following basic facts from the outset:

__Lazy Evaluation:__ Dask employs "lazy evaluation" of jobs: nothing happens until the last possible point in the workflow. For example, a notebook cell may contain a data frame defined by a query 

    df = df.join(df2)

but as far as Dask is concerned this is simply a definiton, or "transformation", of a data resource. If nothing is done with the data frame defined by that query in the cell, then the cell will appear to run instantaneously. Only if an "action" is made on the data frame subsequently will physical execution take place. This means, for example, that if a cell contains a run-time error this may not show up until a later cell is executed with the exception trace explaining the error appearing in that later cell's output (as opposed to any output associated with the cell containing the source of the error). Note also that the statement above, on it's own, will not result in the entire billon+ row, ~100 column, tera-byte scale data set being loaded into memory. Whatever is expressed in subsequent transformations (e.g. SQL "where" clauses) and actions (e.g. aggregations) will define "filter push-down" optimisations that result in only those data needed to action the process being read into memory and processed.  

__Indexing:__ The data hosted on this platform are arranged by `source_id` as the primary unique identifier in Gaia Data Releases. This key should be employed when joining distinct data sets, e.g. pre-computed best neighbours from external survey catalogues - see the "Data holdings" notebook. It can also be used to pull out specific records quickly:

    gaia_source_df[gaia_source_df["source_id"] == 1234]
   
Otherwise, the general RDBMS paradigm employing SQL indexes to optimise execution plans of queries predicated on certain columns has no equivalent here. It is worth noting however that the combination of "filter push-down" and the underlying storage format ([Parquet](https://parquet.apache.org/docs/)) result in only those data that are needed being read from disk on this platform. This is in contrast to RDBMS where columns are generally read from pages within disk files regardless of need unless covered by an index employed as part of an optimised execution plan. This is one of the ways in which Dask achieves much faster performance in "big data" analysis.

__Distributed processing model:__ Fast performance in handling very large data sets is achieved via cluster parallelism with actions split up and optimised for execution on concurrent workers. This is achievable only if the user's processing works with the Dask data frame. Note that issuing a `DataFrame.compute()`, for example, retrieves the distributed components of the Dask data frame to a single driver process and subsequent actions will be no longer distributed over the cluster of workers. Care should be taken in the final stages of work flows to avoid `OutOfMemory` exceptions by collecting too large a results set to the driver process.



In addition to the example notebooks provided here and the linked resources therein see the following for further information:

* [Dask docs](https://docs.dask.org/en/stable/)

